# Skills Assessment
The IMDB dataset introduced by Maas et al. (2011) provides a collection of movie reviews extracted from the Internet Movie Database, annotated for sentiment analysis. It includes 50,000 reviews split evenly into training and test sets, and its carefully curated mixture of positive and negative examples allows researchers to benchmark and improve various natural language processing techniques. The IMDB dataset has influenced subsequent work in developing vector-based word representations and remains a popular baseline resource for evaluating classification performance and model architectures in sentiment classification tasks (Maas et al., 2011).

Your goal is to train a model that can predict whether a movie review is positive (1) or negative (0). You can download the dataset from the question, or from here.

Out of interest, these exact same techniques can be applied into things such as text moderation for instance.

## IMDB Dataset

### Downloading the dataset

In [1]:
import requests
import zipfile
import io

# URL of the dataset
url = "https://academy.hackthebox.com/storage/modules/292/skills_assessment_data.zip"

# Download the dataset
response = requests.get(url)
if response.status_code == 200:
    print("Download successful")
else:
    print("Failed to download the dataset")

Download successful


### Extracting the dataset

In [3]:
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall("imdb_collection")
    print("Extraction successful")

Extraction successful


### Verifying the extraction

In [4]:
# Verify the extracted files
import os

# List the extracted files
extracted_files = os.listdir("imdb_collection")
print("Extracted files:", extracted_files)

Extracted files: ['test.json', 'train.json']


## Loading the dataset

In [5]:
import pandas as pd

train_df = pd.read_json("imdb_collection/train.json")
test_df = pd.read_json("imdb_collection/test.json")

In [7]:
display(test_df)

,text,label
0,I went and saw this movie last night after bei...,1
1,Actor turned director Bill Paxton follows up h...,1
2,As a recreational golfer with some knowledge o...,1
3,"I saw this film in a sneak preview, and it is ...",1
4,Bill Paxton has taken the true story of the 19...,1
...,...,...
24995,I occasionally let my kids watch this garbage ...,0
24996,When all we have anymore is pretty much realit...,0
24997,The basic genre is a thriller intercut with an...,0
24998,Four things intrigued me as to this film - fir...,0


### Displaying Basic Information

In [ ]:
# Display basic information about the dataset

print("-------------------- HEAD --------------------")
print(train_df.head())
print("-------------------- DESCRIBE --------------------")
print(train_df.describe())
print("-------------------- INFO --------------------")
print(train_df.info())

print("-------------------- HEAD --------------------")
print(test_df.head())
print("-------------------- DESCRIBE --------------------")
print(test_df.describe())
print("-------------------- INFO --------------------")
print(test_df.info())

### Basic Sanity Checks

In [9]:
# Check for missing values
print("Missing values:\n", train_df.isnull().sum())
# Check for duplicates
print("Duplicate entries:", train_df.duplicated().sum())

# Check for missing values
print("Missing values:\n", test_df.isnull().sum())
# Check for duplicates
print("Duplicate entries:", test_df.duplicated().sum())

Missing values:
 text     0
label    0
dtype: int64
Duplicate entries: 96
Missing values:
 text     0
label    0
dtype: int64
Duplicate entries: 199


### Removing Any Duplicates

In [10]:
train_df = train_df.drop_duplicates()
test_df = test_df.drop_duplicates()

# Check for missing values
print("Missing values:\n", train_df.isnull().sum())
# Check for duplicates
print("Duplicate entries:", train_df.duplicated().sum())

# Check for missing values
print("Missing values:\n", test_df.isnull().sum())
# Check for duplicates
print("Duplicate entries:", test_df.duplicated().sum())

Missing values:
 text     0
label    0
dtype: int64
Duplicate entries: 0
Missing values:
 text     0
label    0
dtype: int64
Duplicate entries: 0


## Preprocessing the IMDB dataset

### Download the necessary data files

In [11]:
import nltk  # Natural Language Toolkit — Python's standard library for text processing

# Download the necessary NLTK data files (one-time setup; cached locally after first run)
nltk.download("punkt")        # Pre-trained tokenizer models for splitting text into words/sentences
nltk.download("punkt_tab")    # Newer table-based tokenizer data (required by NLTK 3.8.2+)
nltk.download("stopwords")    # List of common words (e.g., "the", "is", "and") to filter out as noise

# Print a header so we can visually compare the raw data against later preprocessing stages
print("=== BEFORE ANY PREPROCESSING ===")

# Display the first 5 rows of the dataframe to inspect the raw, unprocessed messages
print(train_df.head(5))
print(test_df.head(5))

=== BEFORE ANY PREPROCESSING ===
                                                text  label
0  Bromwell High is a cartoon comedy. It ran at t...      1
1  Homelessness (or Houselessness as George Carli...      1
2  Brilliant over-acting by Lesley Ann Warren. Be...      1
3  This is easily the most underrated film inn th...      1
4  This is not the typical Mel Brooks film. It wa...      1
                                                text  label
0  I went and saw this movie last night after bei...      1
1  Actor turned director Bill Paxton follows up h...      1
2  As a recreational golfer with some knowledge o...      1
3  I saw this film in a sneak preview, and it is ...      1
4  Bill Paxton has taken the true story of the 19...      1


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Clean and normalize each review by lowercasing, stripping punctuation and numbers, tokenizing into individual words, removing common stopwords, stemming each word to its root form, then rejoining the tokens into a single string ready for vectorization.

### Preprocessing the Datasets

In [13]:
import numpy as np
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

# Preprocess a single review
def preprocess_review(review):
    # 1. Lowercase: "FREE" and "free" should be the same feature.
    review = review.lower()

    # 2. Strip noise: remove anything that isn't a lowercase letter, whitespace,
    #    "$", or "!". Keeping $ and ! on purpose — they're spam signals.
    #    (regex: ^ inside [] means "not these"; \s = whitespace.)
    review = re.sub(r"[^a-z\s$!]", "", review)

    # 3. Tokenize: split the cleaned string into a list of individual words.
    tokens = word_tokenize(review)

    # 4. Drop stopwords: remove low-signal words ("the", "is", "to") using the
    #    same NLTK stopword list applied during training.
    tokens = [word for word in tokens if word not in stop_words]

    # 5. Stem: collapse word variants to a common root ("joking" -> "joke")
    #    with the same Porter stemmer used in training.
    tokens = [stemmer.stem(word) for word in tokens]

    # 6. Rejoin into a single space-separated string — the format the
    #    vectorizer inside the pipeline expects as input.
    return " ".join(tokens)

Apply the preprocessing to both training and testing IMDB datasets:

In [14]:
train_df["text"] = train_df["text"].apply(preprocess_review)
test_df["text"] = test_df["text"].apply(preprocess_review)

print("=== AFTER PREPROCESSING ===")

# Display the first 5 rows of the dataframes to inspect the processed reviews
print(train_df.head(5))
print(test_df.head(5))

=== AFTER PREPROCESSING ===
                                                text  label
0  bromwel high cartoon comedi ran time program s...      1
1  homeless houseless georg carlin state issu yea...      1
2  brilliant overact lesley ann warren best drama...      1
3  easili underr film inn brook cannon sure flaw ...      1
4  typic mel brook film much less slapstick movi ...      1
                                                text  label
0  went saw movi last night coax friend mine ill ...      1
1  actor turn director bill paxton follow promis ...      1
2  recreat golfer knowledg sport histori pleas di...      1
3  saw film sneak preview delight cinematographi ...      1
4  bill paxton taken true stori us golf open made...      1


## Feature Extraction

### Using CountVectorizer for the Bag-of-Words Approach
In spam we had one dataset so we did fit_transform on everything then split later. Here we already have separate train and test sets, so:

- `fit_transform(train_df)` — learns the vocabulary FROM training data AND converts it to vectors
- `transform(test_df)` — converts test data using the SAME vocabulary learned from train, no re-learning

If we did `fit_transform` on test too, it would learn a slightly different vocabulary from test data — that's data leakage, the model would be peeking at test info during training. Always fit on train only.

In [15]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2))

X_train = vectorizer.fit_transform(train_df["text"])  # fit on train only
X_test = vectorizer.transform(test_df["text"])         # transform test with same vocab

y_train = train_df["label"]
y_test = test_df["label"]

## Model Training, Hyperparameter Tuning and Evaluation

In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer

pipeline = Pipeline([
    ("vectorizer", CountVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2))),
    ("classifier", MultinomialNB())
])

param_grid = {"classifier__alpha": [5.0, 10.0, 20.0, 50.0, 100.0]}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy")
grid.fit(train_df["text"], y_train)

print(f"Best params: {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.4f}")

Best params: {'classifier__alpha': 20.0}
Best CV accuracy: 0.8318


### F1 Score Preview

In [18]:
results = pd.DataFrame(grid.cv_results_)
print(results[["param_classifier__alpha", "mean_test_score", "std_test_score", "rank_test_score"]])

   param_classifier__alpha  mean_test_score  std_test_score  rank_test_score
0                      5.0         0.825771        0.008709                5
1                     10.0         0.830148        0.006032                3
2                     20.0         0.831754        0.004779                1
3                     50.0         0.830389        0.004625                2
4                    100.0         0.828582        0.003770                4


### Grab the Best Model

In [21]:
best_model = grid.best_estimator_
print("Best model parameters:", grid.best_params_)

Best model parameters: {'classifier__alpha': 20.0}


In [22]:
display(best_model)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('vectorizer', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


### Model Evaluation on Test Set
Running the trained model against the 25k test reviews it has never seen during training, to get a real-world accuracy score. This is the final check before saving.

In [23]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = best_model.predict(test_df["text"])
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

Test Accuracy: 0.8501
              precision    recall  f1-score   support

    Negative       0.85      0.85      0.85     12361
    Positive       0.85      0.85      0.85     12440

    accuracy                           0.85     24801
   macro avg       0.85      0.85      0.85     24801
weighted avg       0.85      0.85      0.85     24801



### Saving the Model

In [24]:
import joblib
joblib.dump(best_model, "sentiment_model.joblib")
print("Model saved.")

Model saved.


## Evaluating New Reviews

In [27]:
# Example movie reviews for inference

new_reviews = [
    # Positive Sentiments
    "An absolute masterpiece, I would watch it again in a heartbeat!",
    "Honestly, it was way better than I expected. Highly recommend.",
    "The acting was top-tier and the visuals were stunning.",
    "Wow, just wow. The ending left me speechless.",
    "A fun ride from start to finish, perfect for a movie night.",
    "LOVED IT! Best move of the year hands down!!",
    
    # Negative Sentiments
    "Save your money, this was a complete waste of time.",
    "The plot made absolutely no sense and the acting was wooden.",
    "I fell asleep halfway through, so incredibly boring.",
    "WORST MOVE EVER! DO NOT WATCH!",
    "It started out okay but completely fell apart in the second half.",
    "I want a refund for the two hours of my life I just lost.",
    "Terrible pacing, terrible dialogue, just terrible overall.",
    
    # Mixed / Ambivalent Sentiments
    "Great visuals and effects, but the story was pretty weak.",
    "The main actor was fantastic, but the rest of the cast felt flat.",
    "Very good move overall, needs some spicy scenes though",
    "It had its moments, but it dragged on for way too long.",
    "Cool concepts, but the execution just wasn't there.",
    "I didn't hate it, but I definitely didn't love it either.",
    
    # Neutral / Indifferent Sentiments
    "Yeah I guess it's alright",
    "It was okay. Nothing special, but not terrible.",
    "Just your average popcorn flick, forgettable but fine.",
    "It met my expectations, which weren't very high to begin with.",
    "It is what it is, just a standard action move.",
    "Not the best, not the worst. It passed the time."
]

# Preprocess and predict
processed = [preprocess_review(r) for r in new_reviews]
predictions = best_model.predict(processed)

# Display results
for review, label in zip(new_reviews, predictions):
    sentiment = "Positive 😊" if label == 1 else "Negative 😞"
    print(f"{sentiment}: {review}")

Positive 😊: An absolute masterpiece, I would watch it again in a heartbeat!
Positive 😊: Honestly, it was way better than I expected. Highly recommend.
Positive 😊: The acting was top-tier and the visuals were stunning.
Negative 😞: Wow, just wow. The ending left me speechless.
Positive 😊: A fun ride from start to finish, perfect for a movie night.
Positive 😊: LOVED IT! Best move of the year hands down!!
Negative 😞: Save your money, this was a complete waste of time.
Negative 😞: The plot made absolutely no sense and the acting was wooden.
Negative 😞: I fell asleep halfway through, so incredibly boring.
Negative 😞: WORST MOVE EVER! DO NOT WATCH!
Negative 😞: It started out okay but completely fell apart in the second half.
Negative 😞: I want a refund for the two hours of my life I just lost.
Negative 😞: Terrible pacing, terrible dialogue, just terrible overall.
Positive 😊: Great visuals and effects, but the story was pretty weak.
Positive 😊: The main actor was fantastic, but the rest of the